# Constitution du dataset de mots allemands avec article et niveau

Ce notebook combine deux sources de données afin de constituer une base de mots allemands utilisable par l'application :

1. **nouns.csv** (dataset Wiktionary, environ 100 000 mots) qui fournit l'article grammatical (der, die, das) de chaque mot, mais aucune information de niveau.
2. **goethe-institute-wordlist** (Wortlisten officielles du Goethe-Institut, niveaux A1, A2, B1) qui fournit un niveau CECRL fiable pour un sous-ensemble de mots.

L'objectif final est de produire deux fichiers distincts, sans doublons entre eux :

- un fichier contenant les mots pour lesquels un niveau est connu (mot, article, niveau)
- un fichier contenant les mots pour lesquels le niveau reste inconnu (mot, article)

## 1. Sources

**Sources des données**

- [**Dataset des noms allemands avec article et genre (nouns.csv)**](https://github.com/gambolputty/german-nouns) : GitHub, gambolputty/german-nouns, consulté le 13 septembre 2026
- [**Wortlisten du Goethe-Institut, version TSV par niveau (A1, A2, B1)**](https://github.com/ilkermeliksitki/goethe-institute-wordlist) : GitHub, ilkermeliksitki/goethe-institute-wordlist, consulté le 13 septembre 2026

**PDF officiels Goethe-Institut (sources originales des Wortlisten ci-dessus)**

| Niveau | Lien | Date de consultation |
|---|---|---|
| A1 | [Wortliste A1](https://www.goethe.de/pro/relaunch/prf/sw/Goethe-Zertifikat_A1_Fit1_Wortliste.pdf) | 13 septembre 2026 |
| A2 | [Wortliste A2](https://lmsspada.kemdikbud.go.id/mod/resource/view.php?id=108392) | 13 septembre 2026 |
| B1 | [Wortliste B1](https://www.goethe.de/pro/relaunch/prf/da/Goethe-Zertifikat_B1_Wortliste.pdf) | 13 septembre 2026 |

## 2. Imports

In [10]:
import re
import glob

import pandas as pd

## 3. Chargement du dataset nouns.csv (mots avec article, sans niveau)

In [11]:
# Chemins des fichiers sources
nouns_path = '../raw_data/nouns.csv'
goethe_institute_path = '../raw_data/goethe-institute-wordlist-main'

# Correspondance entre le code de genre du dataset Wiktionary et l'article allemand correspondant
genus_to_article = {"m": "der", "f": "die", "n": "das"}

# Chargement du dataset Wiktionary
# low_memory = False évite l'avertissement de type mixte sur certaines colonnes de déclinaison,
# qui ne sont pas utilisées ici
df_nouns = pd.read_csv(nouns_path, low_memory = False)

print("Nombre total de lignes :", len(df_nouns))
print("Nombre de mots uniques (lemma) :", df_nouns["lemma"].nunique())

Nombre total de lignes : 102444
Nombre de mots uniques (lemma) : 100063


In [12]:
def get_article(mot, df=df_nouns):
    """
    Recherche l'article grammatical d'un mot allemand dans le dataset Wiktionary.
    Retourne None si le mot n'est pas trouvé.
    """
    row = df[df["lemma"] == mot]
    if row.empty:
        return None
    genus = row.iloc[0]["genus"]
    return genus_to_article.get(genus)


# Vérification rapide sur deux mots connus
print(get_article("Hund"))   # attendu : der
print(get_article("Tisch"))  # attendu : der

der
der


## 4. Exploration d'un fichier Wortliste Goethe-Institut

In [13]:
# Inspection d'un fichier exemple afin de comprendre la structure des Wortlisten
df_exemple = pd.read_csv(f"{goethe_institute_path}/a1/a.tsv", sep="\t", header=None)

print(df_exemple.shape)
print(df_exemple.head(15))

(92, 3)
                  0                                                  1  \
0                ab                       Ab morgen muss ich arbeiten.   
1              aber  Ich bin oft im Büro, aber nur für wenige Stunden.   
2          abfahren                        Wir fahren um zwölf Uhr ab.   
3       die Abfahrt                       Vor der Abfahrt rufe ich an.   
4           abgeben                  Ich muss meine Schlüssel abgeben.   
5        abholen(1)         Wann kann ich den Schrank bei dir abholen?   
6        abholen(2)             Wir müssen noch meinen Bruder abholen.   
7      der Absender           Da ist ein Brief für dich ohne Absender.   
8           Achtung                 Achtung! Das dürfen Sie nicht tun.   
9   die Adresse,-en                Können Sie mir seine Adresse sagen?   
10          all-(1)                                        Alles Gute!   
11          all-(2)                                     Das ist alles.   
12          all-(3)           

Chaque ligne contient trois colonnes : le mot brut (parfois précédé de son article s'il s'agit d'un nom commun), une phrase d'exemple en allemand, et une phrase d'exemple en anglais. Seuls les mots précédés d'un article (der, die, das) sont des noms communs ; ce sont les seuls qui nous intéressent ici.

## 5. Extraction des noms communs avec article et niveau (A1, A2, B1)

In [14]:
def parse_word(raw):
    """
    Extrait l'article et le mot à partir d'une entrée brute de Wortliste.
    Retourne None si l'entrée n'est pas un nom commun (verbe, adverbe, etc.).
    Nettoie également les suffixes de type ",-en" (marque du pluriel)
    et "(1)", "(2)" (numéros de sens multiples).
    """
    raw = raw.strip()
    match = re.match(r"^(der|die|das)\s+(.+)$", raw)
    if not match:
        return None

    article, mot = match.groups()
    mot = re.sub(r",.*$", "", mot)
    mot = re.sub(r"\(\d+\)$", "", mot)
    return article, mot.strip()


# Parcours des trois niveaux disponibles dans le repo Goethe-Institut
lignes = []
for niveau in ["a1", "a2", "b1"]:
    fichiers = glob.glob(f"{goethe_institute_path}/{niveau}/*.tsv")
    for fichier in fichiers:
        df_niveau = pd.read_csv(
            fichier, sep="\t", header=None, names=["mot_brut", "exemple_de", "exemple_en"]
        )
        for _, row in df_niveau.iterrows():
            parsed = parse_word(row["mot_brut"])
            if parsed:
                article, mot = parsed
                lignes.append({"mot": mot, "article": article, "niveau": niveau.upper()})

# Constitution du dataframe final, sans doublons
df_final = pd.DataFrame(lignes).drop_duplicates(subset=["mot"])

print(df_final.shape)
print(df_final.head(15))

(1621, 3)
                 mot article niveau
0            Abfahrt     die     A1
1           Absender     der     A1
2            Adresse     die     A1
3              Alter     das     A1
4            Angebot     das     A1
5             Anfang     der     A1
7            Ankunft     die     A1
8          Anmeldung     die     A1
9             Anrede     die     A1
10             Anruf     der     A1
11  Anrufbeantworter     der     A1
12            Ansage     die     A1
13         Anschluss     der     A1
15           Antwort     die     A1
16           Anzeige     die     A1


In [15]:
# Répartition du nombre de mots par niveau
print(df_final["niveau"].value_counts())

niveau
B1    980
A1    327
A2    314
Name: count, dtype: int64


In [16]:
# Sauvegarde du fichier intermédiaire issu des Wortlisten Goethe-Institut
df_final.to_csv(f"{goethe_institute_path}/mots_avec_article_niveau.csv", index=False)

## 6. Fusion des deux sources et séparation finale

In [17]:
# Rechargement des deux fichiers sources pour cette étape de fusion
df_nouns = pd.read_csv(nouns_path, low_memory=False)
df_final = pd.read_csv(f"{goethe_institute_path}/mots_avec_article_niveau.csv")

# Préparation de nouns.csv : une seule ligne par mot unique, avec son article
df_nouns_clean = df_nouns.drop_duplicates(subset=["lemma"]).copy()
df_nouns_clean["article"] = df_nouns_clean["genus"].map(genus_to_article)
df_nouns_clean = df_nouns_clean.rename(columns={"lemma": "mot"})[["mot", "article"]]

# Mots présents dans les deux fichiers : ceux-là disposent d'un niveau connu et fiable
mots_avec_niveau = df_final[df_final["mot"].isin(df_nouns_clean["mot"])].copy()

# Mots de nouns.csv qui n'ont pas de correspondance dans les Wortlisten : niveau inconnu
mots_sans_niveau = df_nouns_clean[~df_nouns_clean["mot"].isin(df_final["mot"])].copy()

print("Mots avec niveau :", mots_avec_niveau.shape)
print("Mots sans niveau :", mots_sans_niveau.shape)

Mots avec niveau : (1499, 3)
Mots sans niveau : (98565, 2)


In [18]:
# Sauvegarde des deux fichiers finaux, sans doublons entre eux
mots_avec_niveau.to_csv(f"{goethe_institute_path}/mots_avec_niveau_final.csv", index=False)
mots_sans_niveau.to_csv(f"{goethe_institute_path}/mots_sans_niveau_final.csv", index=False)

## 7. Points restants à traiter

- Vérifier les mots présents dans les Wortlisten mais absents de nouns.csv (environ 122 mots), afin de ne pas les perdre.
- Ajouter la traduction française pour chacun des deux fichiers finaux.

## 8. Méthodologie de traduction

Les traductions française et anglaise du fichier `nouns.csv` ont été générées via Google Sheets, avec la fonction native `GOOGLETRANSLATE`, à partir de la colonne `mot` (allemand) :

- Traduction française : `=GOOGLETRANSLATE(A2, "de", "fr")`
- Traduction anglaise : `=GOOGLETRANSLATE(A2, "de", "en")`

Ces traductions sont générées automatiquement à partir du mot isolé, sans contexte de phrase. Elles couvrent correctement l'immense majorité du vocabulaire courant (A1/A2/B1), mais peuvent être imprécises ou incomplètes pour les mots polysémiques (plusieurs sens selon le contexte) ou les mots très spécifiques. Une relecture manuelle ciblée sur ces cas reste recommandée avant une mise en production finale.

Une fois le niveau des mots restants (actuellement sans niveau CECRL connu) déterminé, ces mots suivront la même structure de colonnes (`mot, article, niveau, traduction_fr, traduction_en`) et pourront être chargés dans la base de données relationnelle sans rupture de format ni interruption du pipeline existant.